# 05 — Descriptive and Communication Metrics

**Project:** From Online Attention to Electoral Outcomes: A Network Science Analysis of Philippine Election 2025 Twitter/X Communication

This notebook is part of the notebook set for the Philippine Election 2025 Twitter/X network science project.



## Visible progress tracker

This notebook now prints numbered stage updates while it runs. If Jupyter shows `[*]`, the current stage is still processing. A run log is also saved under `outputs/run_logs/`.


In [ ]:
# VISIBLE PROGRESS TRACKER — automatically added in v5
from pathlib import Path as _ProgressPath
import sys as _progress_sys
_PROGRESS_ROOT = _ProgressPath.cwd().parent if _ProgressPath.cwd().name == "notebooks" else _ProgressPath.cwd()
_progress_sys.path.append(str(_PROGRESS_ROOT / "src"))
try:
    from election_network_progress import make_tracker
    progress = make_tracker("05_descriptive_communication_metrics", total_steps=7, root=_PROGRESS_ROOT)
except Exception as _progress_error:
    print(f"Progress tracker could not start: {_progress_error}")
    class _FallbackProgress:
        def __init__(self): self.current = 0
        def step(self, label):
            self.current += 1
            print(f"[stage {self.current}] {label}", flush=True)
        def info(self, label): print(f"[info] {label}", flush=True)
        def done(self, label="Notebook completed"): print(f"✓ {label}", flush=True)
    progress = _FallbackProgress()


## Purpose

This notebook creates non-network communication analytics: activity, engagement, language, candidate visibility, hashtag activity, verified activity, and high-impact posts.

In [ ]:
progress.step("Purpose")
from pathlib import Path
import sys
import warnings
warnings.filterwarnings("ignore")

# Make ../src importable when running from notebooks/
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(ROOT / "src"))

from election_network_utils import *
paths = ensure_dirs(ROOT)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_rows", 80)
px.defaults.template = "plotly_white"

RAW = paths["raw"]
PROCESSED = paths["processed"]
FIGURES = paths["figures"]
INTERACTIVE = paths["interactive"]
TABLES = paths["tables"]
NETWORKS = paths["networks"]
REPORT_ASSETS = paths["report_assets"]

print("Project root:", ROOT)


In [ ]:
progress.step("Purpose")
tweets = pd.read_csv(PROCESSED / "tweets_with_entities.csv", parse_dates=["createdAt"])
tweets = parse_list_columns(tweets, ["hashtags", "mentions", "domains", "candidates_mentioned", "topics_detected"])
print(tweets.shape)
tweets.head()


In [ ]:
progress.step("Purpose")
# Daily candidate mentions
candidate_daily_rows = []
for _, row in tweets.iterrows():
    for c in row["candidates_mentioned"]:
        candidate_daily_rows.append({"date": row["date"], "candidate": c, "count": 1})

candidate_daily = pd.DataFrame(candidate_daily_rows)
if not candidate_daily.empty:
    candidate_daily = candidate_daily.groupby(["date", "candidate"], as_index=False)["count"].sum()
    candidate_daily.to_csv(PROCESSED / "candidate_mentions_by_date.csv", index=False)
    top_candidates = candidate_daily.groupby("candidate")["count"].sum().sort_values(ascending=False).head(10).index
    fig = px.line(candidate_daily[candidate_daily["candidate"].isin(top_candidates)], x="date", y="count", color="candidate", title="Candidate mention timeline: top 10 detected candidates")
    fig.update_layout(height=620, xaxis_title="Date", yaxis_title="Mentions")
    save_plotly(fig, INTERACTIVE / "05_candidate_mention_timeline.html", FIGURES / "05_candidate_mention_timeline.png")
    fig.show()
else:
    print("No candidate mentions were detected. Check candidate dictionary.")


In [ ]:
progress.step("Purpose")
# Engagement by candidate
engagement_cols = ["retweetCount", "replyCount", "likeCount", "quoteCount", "viewCount", "bookmarkCount", "total_engagement"]
rows = []
for _, row in tweets.iterrows():
    for c in row["candidates_mentioned"]:
        item = {"candidate": c}
        for metric in engagement_cols:
            item[metric] = row.get(metric, 0)
        rows.append(item)

candidate_engagement = pd.DataFrame(rows)
if not candidate_engagement.empty:
    candidate_engagement_summary = candidate_engagement.groupby("candidate", as_index=False)[engagement_cols].sum()
    candidate_engagement_summary["tweet_mentions"] = candidate_engagement.groupby("candidate").size().values
    candidate_engagement_summary = candidate_engagement_summary.sort_values("total_engagement", ascending=False)
    candidate_engagement_summary.to_csv(PROCESSED / "candidate_engagement_summary.csv", index=False)
    display(candidate_engagement_summary.head(20))

    fig = px.scatter(candidate_engagement_summary, x="tweet_mentions", y="total_engagement", size="viewCount", hover_name="candidate", title="Candidate mentions vs total engagement")
    fig.update_layout(height=620, xaxis_title="Candidate mentions", yaxis_title="Total engagement")
    save_plotly(fig, INTERACTIVE / "05_candidate_mentions_vs_engagement.html", FIGURES / "05_candidate_mentions_vs_engagement.png")
    fig.show()
else:
    print("No candidate engagement summary generated.")


In [ ]:
progress.step("Purpose")
# Top high-impact tweets
high_impact = tweets.sort_values("total_engagement", ascending=False).head(100)
high_impact[["createdAt", "lang", "total_engagement", "viewCount", "likeCount", "retweetCount", "quoteCount", "text", "candidates_mentioned", "hashtags"]].to_csv(TABLES / "05_top100_high_impact_tweets.csv", index=False)
high_impact[["createdAt", "total_engagement", "viewCount", "likeCount", "retweetCount", "text"]].head(20)


In [ ]:
progress.step("verified_summary = tweets.groupby('author_isBlueVerified', as_index=False).agg(")
# Verified vs non-verified comparison
verified_summary = tweets.groupby("author_isBlueVerified", as_index=False).agg(
    tweet_count=("pseudo_id", "count"),
    unique_authors=("pseudo_author_userName", "nunique"),
    total_views=("viewCount", "sum"),
    total_likes=("likeCount", "sum"),
    total_retweets=("retweetCount", "sum"),
    avg_views=("viewCount", "mean"),
    avg_total_engagement=("total_engagement", "mean"),
)
verified_summary.to_csv(TABLES / "05_verified_vs_nonverified_summary.csv", index=False)
display(verified_summary)

fig = px.bar(verified_summary, x="author_isBlueVerified", y="tweet_count", title="Tweet activity by Blue Verified status", text="tweet_count")
fig.update_layout(height=460, xaxis_title="Author is Blue Verified", yaxis_title="Tweet count")
save_plotly(fig, INTERACTIVE / "05_verified_tweet_activity.html", FIGURES / "05_verified_tweet_activity.png")
fig.show()


In [ ]:
progress.step("tag_rows = []")
# Hashtag activity timeline for top hashtags
tag_rows = []
for _, row in tweets.iterrows():
    for h in row["hashtags"]:
        tag_rows.append({"date": row["date"], "hashtag": h, "count": 1})

tag_daily = pd.DataFrame(tag_rows)
if not tag_daily.empty:
    tag_daily = tag_daily.groupby(["date", "hashtag"], as_index=False)["count"].sum()
    top_tags = tag_daily.groupby("hashtag")["count"].sum().sort_values(ascending=False).head(10).index
    fig = px.line(tag_daily[tag_daily["hashtag"].isin(top_tags)], x="date", y="count", color="hashtag", title="Top hashtag activity over time")
    fig.update_layout(height=620)
    save_plotly(fig, INTERACTIVE / "05_top_hashtag_timeline.html", FIGURES / "05_top_hashtag_timeline.png")
    fig.show()


## Run complete

The final cell closes the progress bar and confirms that all tracked notebook stages finished.


In [ ]:
progress.done("Notebook run completed")
